In [0]:
#  Install Great Expectations (Run once)
%pip install great-expectations==0.17.23
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.6/813.6 kB 43.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import great_expectations as ge
from pyspark.sql.functions import col, count as sql_count

In [0]:
# Read Data from Bronze Delta Lake
flight_booking_bronze_path = "s3://travel-analytics-bronze/delta/bronze/flight_bookings/"
df_flight_booking = spark.read.format("delta").load(flight_booking_bronze_path)

print("=" * 80)
print("FLIGHT BOOKING VALIDATION WITH GREAT EXPECTATIONS")
print("=" * 80)
print(f"Total records: {df_flight_booking.count()}")
print("\n--- Schema ---")
df_flight_booking.printSchema()

FLIGHT BOOKING VALIDATION WITH GREAT EXPECTATIONS
Total records: 7648

--- Schema ---
root
 |-- _airbyte_ab_id: string (nullable = true)
 |-- _airbyte_emitted_at: timestamp (nullable = true)
 |-- price: long (nullable = true)
 |-- trip_id: long (nullable = true)
 |-- airline_id: long (nullable = true)
 |-- _ab_cdc_lsn: double (nullable = true)
 |-- aircraft_id: string (nullable = true)
 |-- airport_dst: long (nullable = true)
 |-- airport_src: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- seat_number: string (nullable = true)
 |-- arrival_date: date (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- booking_time: string (nullable = true)
 |-- travel_class: string (nullable = true)
 |-- flight_number: string (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- departure_date: date (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |

In [0]:
ge_df = ge.from_pandas(df_flight_booking.toPandas())

print("\nRunning Great Expectations validation...")


Running Great Expectations validation...


In [0]:
# Define and Run Expectations

# Expectation 1: trip_id NOT NULL
result1 = ge_df.expect_column_values_to_not_be_null(column="trip_id")
print(f"✓ trip_id NOT NULL: {result1['success']}")

# Expectation 2: customer_id NOT NULL
result2 = ge_df.expect_column_values_to_not_be_null(column="customer_id")
print(f"✓ customer_id NOT NULL: {result2['success']}")

# Expectation 3: price NOT NULL
result3 = ge_df.expect_column_values_to_not_be_null(column="price")
print(f"✓ price NOT NULL: {result3['success']}")

# Expectation 4: price > 0
result4 = ge_df.expect_column_values_to_be_between(column="price", min_value=1, max_value=None)
print(f"✓ price > 0: {result4['success']}")

# Expectation 5: airport_dst NOT NULL
result5 = ge_df.expect_column_values_to_not_be_null(column="airport_dst")
print(f"✓ airport_dst NOT NULL: {result5['success']}")

# Expectation 6: airport_src NOT NULL
result6 = ge_df.expect_column_values_to_not_be_null(column="airport_src")
print(f"✓ airport_src NOT NULL: {result6['success']}")

✓ trip_id NOT NULL: True
✓ customer_id NOT NULL: True
✓ price NOT NULL: True
✓ price > 0: True
✓ airport_dst NOT NULL: True
✓ airport_src NOT NULL: True


In [0]:
# STEP 6: Apply Validations using PySpark

print("\n--- Applying Validations ---")

# Start with all data
df_valid = df_flight_booking
df_invalid_list = []

# Rule 1: trip_id NOT NULL
df_invalid_trip = df_valid.filter(col("trip_id").isNull())
if df_invalid_trip.count() > 0:
    df_invalid_list.append(df_invalid_trip)
    print(f"  → Found {df_invalid_trip.count()} rows with trip_id = NULL")
df_valid = df_valid.filter(col("trip_id").isNotNull())

# Rule 2: customer_id NOT NULL
df_invalid_customer = df_valid.filter(col("customer_id").isNull())
if df_invalid_customer.count() > 0:
    df_invalid_list.append(df_invalid_customer)
    print(f"  → Found {df_invalid_customer.count()} rows with customer_id = NULL")
df_valid = df_valid.filter(col("customer_id").isNotNull())

# Rule 3: price NOT NULL
df_invalid_price_null = df_valid.filter(col("price").isNull())
if df_invalid_price_null.count() > 0:
    df_invalid_list.append(df_invalid_price_null)
    print(f"  → Found {df_invalid_price_null.count()} rows with price = NULL")
df_valid = df_valid.filter(col("price").isNotNull())

# Rule 4: price > 0
df_invalid_price_negative = df_valid.filter(col("price") <= 0)
if df_invalid_price_negative.count() > 0:
    df_invalid_list.append(df_invalid_price_negative)
    print(f"  → Found {df_invalid_price_negative.count()} rows with price <= 0")
df_valid = df_valid.filter(col("price") > 0)

# Rule 5: airport_dst NOT NULL
df_invalid_dst = df_valid.filter(col("airport_dst").isNull())
if df_invalid_dst.count() > 0:
    df_invalid_list.append(df_invalid_dst)
    print(f"  → Found {df_invalid_dst.count()} rows with airport_dst = NULL")
df_valid = df_valid.filter(col("airport_dst").isNotNull())

# Rule 6: airport_src NOT NULL
df_invalid_src = df_valid.filter(col("airport_src").isNull())
if df_invalid_src.count() > 0:
    df_invalid_list.append(df_invalid_src)
    print(f"  → Found {df_invalid_src.count()} rows with airport_src = NULL")
df_valid = df_valid.filter(col("airport_src").isNotNull())



--- Applying Validations ---


In [0]:
# Combine Invalid Records

if df_invalid_list:
    df_invalid = df_invalid_list[0]
    for df_temp in df_invalid_list[1:]:
        df_invalid = df_invalid.union(df_temp)
    df_invalid = df_invalid.distinct()
else:
    df_invalid = spark.createDataFrame([], df_flight_booking.schema)

valid_count = df_valid.count()
invalid_count = df_invalid.count()
total_count = df_flight_booking.count()

print("\n" + "=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)
print(f"✅ Valid records:   {valid_count} ({valid_count/total_count*100:.2f}%)")
print(f"❌ Invalid records: {invalid_count} ({invalid_count/total_count*100:.2f}%)")



VALIDATION RESULTS
✅ Valid records:   7648 (100.00%)
❌ Invalid records: 0 (0.00%)


In [0]:
# Write Invalid Records to Quarantine

if invalid_count > 0:
    quarantine_path = "s3://travel-analytics-bronze/Quarantine/Flight_Bookings"
    
    df_invalid.write \
        .format("parquet") \
        .mode("append") \
        .save(quarantine_path)
    
    print(f"\n❌ Invalid records sent to Quarantine: {quarantine_path}")
    print("\n--- Sample Invalid Records ---")
    df_invalid.show(10, truncate=False)
else:
    print(f"\n✅ All {valid_count} records passed validation!")

print("\n" + "=" * 80)
print("✅ VALIDATION COMPLETED!")
print("=" * 80)
print(f"Valid records remain in Bronze: {flight_booking_bronze_path}")
if invalid_count > 0:
    print(f"Invalid records in Quarantine: s3://travel-analytics-bronze/Quarantine/Flight_Bookings")


✅ All 7648 records passed validation!

✅ VALIDATION COMPLETED!
Valid records remain in Bronze: s3://travel-analytics-bronze/delta/bronze/flight_bookings/
